In [1]:
import numpy as np
from casadi import *
x = MX.sym("x")
print(jacobian(sin(x),x))


cos(x)


In [ ]:
## Class for Manipulator Dynamics
class Manipulator:

    def __init__(self, M1, M2, L1, L2, init_theta1, init_theta2, final_theta1, final_theta2, max_iter):

        # Manipulator parameters
        self.M1 = M1
        self.M2 = M2
        self.L1 = L1
        self.L2 = L2
        self.init_theta1 = init_theta1
        self.init_theta2 = init_theta2
        self.final_theta1 = final_theta1
        self.final_theta2 = final_theta2
        self.max_iter = max_iter

        # Define simulation parameters
        self.N = 100
        self.g = 9.81
        

        #Initialize optimizer object
        self.opti = Opti()

        # Define arrays for system matrices
        self.M = self.opti.variable(2, 2, self.N+1)
        self.c = self.opti.variable(2, 1, self.N+1)
        self.G = self.opti.variable(2, 1, self.N+1)

        self.X = self.opti.variable(6,self.N+1) # state trajectory
        self.theta1       = self.X[0,:]
        self.theta2       = self.X[1,:]
        self.theta1_dot   = self.X[2,:]
        self.theta2_dot   = self.X[3,:]
        self.theta1_d_dot = self.X[4,:]
        self.theta2_d_dot = self.X[5,:]

        self.T = self.opti.variable()
        
        # Define our decision variables
        self.tau = self.opti.variable(2, self.N+1)        # tau = [tau1, tau2].T

    def make_M(self):
        # Sets up the array of mass matrices
        # for k in range(0, self.N):
            # theta1, theta2 = self.X[0, k], self.X[1, k]
        self.M[0, 0, :] = (self.M1 + self.M2) * self.L1**2
        self.M[0, 1, :] = self.M2 * self.L1 * self.L2 * cos(self.theta1 - self.theta2)
        self.M[1, 0, :] = self.M2 * self.L1 * self.L2 * cos(self.theta1 - self.theta2)
        self.M[1, 1, :] = self.M2 * self.L2**2
        return

    def make_c(self):
        # for k in range(0, self.N):
            # theta1, theta2, theta1dot, theta2dot = self.X[0, k], self.X[1, k], self.X[2, k], self.X[3, k]
        self.c[0, 0, :] = self.M2*self.L1*self.L2*self.theta1_dot**2 * sin(self.theta1-self.theta2)
        self.c[1, 0, :] = -self.M2*self.L1*self.L2*self.theta1_dot**2 * sin(self.theta1-self.theta2)
        return

    def make_G(self):
        # for k in range(0, self.N):
            # theta1, theta2 = self.X[0, k], self.X[1, k]
        self.G[0, 0, :] = (self.M1+self.M2) * self.g * self.L1 * cos(self.theta1)
        self.G[1, 0, :] = self.M2 *self.g *self.L2 * cos(self.theta2)
        return

    def diff(self, k):
        # Returns the time derivative of X as defined in the doc, at timestep k
        for k in range(0, self.n):
            self.Xdot[0:2, k] = self.X[2:4, k]
            B = self.tau[:, k] - self.c[:, k] - self.G[:, k]
            M_inv = solve(self.M[:, :, k], MX.eye(2))
            self.Xdot[2:4, k] = M_inv @ B
        return

    def set_initial_states(self):
        # Define starting conditions
        self.opti.subject_to(self.theta1[0]==self.init_theta1)
        self.opti.subject_to(self.theta2[0]==self.init_theta2)
        self.opti.subject_to(self.theta1[self.N]==self.final_theta1)
        self.opti.subject_to(self.theta2[self.N]==self.final_theta2)
        return

    def build_objectives(self):
        # Define objective function to minimize
        #K = 1/2 I * omega^2
        #P = mgh
        KE = 0.5 * self.M[0, 0, :] * self.theta1_dot ** 2 + 0.5 * self.M[1, 1, :] * self.theta2_dot ** 2 + self.M[0, 1, :]
        PE = (self.M1+self.M2) * self.g * self.L1 * sin(self.theta1) + self.M2 * self.g * self.L2 * sin(self.theta2)
        objective = KE - PE
        self.opti.minimize(objective)
        # opti.minimize(T)
        return

    def build_equality_constraints(self):
        # Build constraints as defined in the doc
        self.opti.subject_to(self.T>=0)
        # for k in range(N - 1):
        #     opti.subject_to(q1[k+1] == q1[k] + dt * dq1[k])
        #     opti.subject_to(q2[k+1] == q2[k] + dt * dq2[k])
        return

    def solve_optimizer(self):
        p_opts = {"record_time":True}
        s_opts = {"max_iter": self.max_iter}
        self.opti.solver('ipopt', p_opts, s_opts)
        self.set_initial_states()
        self.build_objectives()
        self.build_equality_constraints()
        sol = self.opti.solve()
        return sol    